In [ ]:
import shutil
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import PIL
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import torchvision
from IPython.display import Video
from torchvision import transforms, datasets
from torchvision.io import read_image
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid
from torch.utils.data import DataLoader


In [ ]:
# setting the device
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
print(f'using {device} device')

In [ ]:
# creating the directory for the project
project_dir = Path(r'C:\Users\HP\Documents\WQU AI\projectface1')
project_dir.mkdir(exist_ok=True)

In [ ]:
data_dir = 'data'
video_dir = project_dir / data_dir
video_dir.mkdir(exist_ok=True)

print(video_dir)

In [ ]:
# getting the video(it was in another directory)
vid_folder = r'C:\Users\HP\Documents\projvid'

In [ ]:
import glob
import os
videos = glob.glob(os.path.join(vid_folder, '*.mp4'))
for i, video in enumerate(videos):
    globals()[f'input_video{i}'] = video

print(input_video1)

In [ ]:
# creating a path for the combined video
output_video = 'output.mp4'
combined_video = video_dir / output_video

In [ ]:
# with open('videos.txt', 'w') as f:
#     for video in videos:
#         f.write(f'file "{video}"\n')

# subprocess.run([
#     'ffmpeg', '-f', 'concat', '-safe', '0', '-i', 'videos.txt', '-c', 'copy', 'combined_video'
# ])


In [ ]:
from moviepy import VideoFileClip, concatenate_videoclips

def standardize_video(video_path):
    clip = VideoFileClip(video_path)
    # Ensure consistent format
    return clip

video1 = standardize_video(input_video0)
video2 = standardize_video(input_video1)
video3 = standardize_video(input_video2)
video4 = standardize_video(input_video3)

# Use method='compose' for better compatibility
final_video = concatenate_videoclips([video1, video2, video3, video4], method='compose')
final_video.write_videofile('combined_video.mp4', codec='libx264')

In [ ]:
type(final_video)

In [ ]:
from IPython.display import Video
Video('combined_video.mp4', width=400,height=400, embed=True)

# Exploring and Preparing the data

In [ ]:
stanvid = 'combined_video.mp4'

In [ ]:
video_capture = cv2.VideoCapture(stanvid)

if not video_capture.isOpened():
    print('Error: Could not open Video.')
else:
    frame_rate = video_capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    print(frame_count, frame_rate)

In [ ]:
ret, first_frame = video_capture.read()

if ret:
    plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
    plt.title('First Frame')
    plt.axis('off')
    plt.show()
else:
    print('Couldn\'t read frame')

In [ ]:
height, width, channel = first_frame.shape
print(height, width, channel)

In [ ]:
# directory for storing the extracted frames
frames_dir = video_dir / 'extracted_frames'

frames_dir.mkdir(exist_ok=True)

In [ ]:
print(frames_dir)

In [ ]:
# lets get every second frame
interval = frame_rate * 0.20
frame_count = 0

print('Start extracting individual frames...')
while True:
    # read the next frame from the video capture
    ret, frame = video_capture.read()
    if not ret:
        print('Finished!!')
        break

    if frame_count % interval == 0:
        frame_path = frames_dir / f'frame_{frame_count}.jpg'
        cv2.imwrite(frame_path, frame)

    frame_count += 1

video_capture.release()

In [ ]:
# no of frames extracted
n_extracted_frames = len(list(frames_dir.iterdir()))

print(n_extracted_frames)

In [ ]:
# # function to display a sample of the frames
# def display_sample_images(dir_path, sample=5):
#     image_list = []
#     images = sorted(dir_path.iterdir())
#     if images:
#         sample_images = images[:sample]
#         for sample_image in sample_images:
#             image = read_image(str(sample_image))
#             # resizing transformation
#             resize_transform = transforms.Resize((240, 240))
#             image = resize_transform(image)
#             image_list.append(image)
#     grid = make_grid(image_list, nrow=5)
#     image = to_pil_image(grid)
#     return image

In [ ]:
# display_sample_images(frames_dir, sample=15)

## Modelling

In [ ]:
# initializing the MTCNN model
MTCNN?

In [ ]:
mtcnn = MTCNN(device=device, keep_all=True, min_face_size=60, post_process=False, select_largest=False, selection_method='center_weighted')
print(mtcnn)

In [ ]:
# file path for sample image
sample_image_filename = 'frame_516.jpg'
sample_image_path = frames_dir / sample_image_filename

sample_image = PIL.Image.open(sample_image_path)
sample_image

In [ ]:
# using the correct file format
sample_image = cv2.cvtColor(cv2.imread(str(sample_image_path)), cv2.COLOR_BGR2RGB)

In [ ]:
sample_image_filename2 = 'frame_1056.jpg'
sample_image_path2 = frames_dir / sample_image_filename2
sample_image2 = cv2.cvtColor(cv2.imread(str(sample_image_path2)), cv2.COLOR_BGR2RGB)

### Detecting using bounding boxes

In [ ]:
# to check how many faces were detectes and th prob of detecting
boxes, prob = mtcnn.detect(sample_image)

In [ ]:
print(boxes)
print(boxes.shape)

In [ ]:
number_of_detected_faces = len(boxes)
number_of_detected_faces

In [ ]:
# faces that the model is confidebt is a face
num_faces = len(prob[prob>0.95])
num_faces

In [ ]:
# plotting the bounding boxes
fig, ax = plt.subplots()
ax.imshow(sample_image)

for box in boxes:
    rect = plt.Rectangle(
        (box[0], box[1]), box[2] - box[0], box[3] - box[1], fill = False, color = 'red'
    )
    ax.add_patch(rect)
plt.axis('off');

## Extracting Facial Landmarks


In [ ]:
boxes, prob, landmarks = mtcnn.detect(sample_image, landmarks=True)

In [ ]:
boxes2,prob2,  landmarks2 = mtcnn.detect(sample_image2, landmarks=True)

In [ ]:
print(boxes)
print(f'landmarks: {landmarks}')

In [ ]:
print(landmarks.shape)

In [ ]:
# Plot with proper coordinate handling
fig, ax = plt.subplots()
ax.imshow(sample_image2)

if boxes is not None:
    for box in boxes2:
        rect = plt.Rectangle(
            (box[0], box[1]), 
            box[2] - box[0], 
            box[3] - box[1], 
            fill=False, color="blue", linewidth=2
        )
        ax.add_patch(rect)

if landmarks is not None:
    for landmark in landmarks2:
        for point in landmark:
            ax.plot(point[0], point[1], marker="o", color="red", markersize=4)

plt.axis("off")
plt.show()

In [ ]:
sample_image = cv2.cvtColor(cv2.imread(str(sample_image_path)), cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots()
ax.imshow(sample_image)

for box in boxes:
    rect = plt.Rectangle(
        (box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, color="red"
    )
    ax.add_patch(rect)
for landmark in landmarks:
    for point in landmark:
        ax.plot(point[0], point[1], marker="o", color="blue")
plt.axis("off");

In [ ]:
# cropping out detected faces
faces = mtcnn(sample_image)

print(faces.shape)

In [ ]:
# grid of the detected faces if more than one

Grid = make_grid(faces, nrow=1)

print(Grid.shape)

In [ ]:
# plotting
plt.imshow(Grid.permute(1, 2, 0).int())
plt.axis('off')

In [ ]:
# directory for created images

images_dir = video_dir / 'images'
images_dir.mkdir(exist_ok = True)

In [ ]:
stan_dir = images_dir / 'stanley'

stan_dir.mkdir(exist_ok=True)

In [ ]:
stanley_imgs = [f'frame_{i}.jpg' for i in [456, 465, 615, 612, 702, 930, 1080, 1092, 1062, 1020, 936]]

# hand selecting good images

# for i in range(444, 1123, 6):
#     stanley_imgs.append(f'frame_{i}.jpg')
#     i =+  6

In [ ]:
len(stanley_imgs)


In [ ]:
stanley_imgs_paths = [frames_dir / img for img in stanley_imgs]

In [ ]:
len(stanley_imgs_paths)

In [ ]:
# plot the images
fig, axs = plt.subplots(2, 5, figsize=(10, 8))
axs = axs.flatten()

for i, ax in enumerate(axs):
    ax.imshow(PIL.Image.open(stanley_imgs_paths[i]))
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
for image_path in stanley_imgs_paths:
    shutil.copy(image_path, stan_dir)

In [ ]:
len(list(stan_dir.iterdir()))

In [ ]:
# for timmie dawg
timz_dir = images_dir / 'timmie'

timz_dir.mkdir(exist_ok=True)

In [ ]:
timz_images = [f'frame_{i}.jpg' for i in [6, 75, 78, 180, 198, 36, 150, 168,0, 6]]

In [ ]:
timz_img_paths = [frames_dir / img for img in timz_images]

len(timz_images)

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(10, 8))
axs = axs.flatten()

for i, ax in enumerate(axs):
    ax.imshow(PIL.Image.open(timz_img_paths[i]))
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
for image_path in timz_img_paths:
    shutil.copy(image_path, timz_dir)

len(list(timz_dir.iterdir()))

In [ ]:
# this model detects only one face
mtcnn0 = MTCNN(image_size=240, keep_all=False, min_face_size=40)


In [ ]:
weight_path = r'c:\Users\HP\Downloads\20180402-114759-vggface2 (1).pt'

In [ ]:
# Let the library download the correct weights automatically
# First, fix SSL issues if any
import ssl
import urllib3
urllib3.disable_warnings()
ssl._create_default_https_context = ssl._create_unverified_context

# Load with pretrained=True (uses correct vggface2 weights)
resnet = InceptionResnetV1(pretrained='vggface2').eval()
print("Model loaded with automatic weights!")

In [ ]:
print(images_dir)

In [ ]:
# creating an image folder object
dataset = datasets.ImageFolder(images_dir)

print(dataset)

In [ ]:
# now the subdirectories are considered as classes
for subdirectory in images_dir.iterdir():
    print(subdirectory)

In [ ]:
dataset.class_to_idx.items()

In [ ]:
idx_to_class = {id: name for name, id in dataset.class_to_idx.items()}

print(idx_to_class)


In [ ]:
# define a collate function that returns the first element of atuple, which is the image object
def collate_fn(x):
    return x[0]

In [ ]:
# constructing a dataloader object
loader = DataLoader(dataset, collate_fn=collate_fn)
print(loader.dataset)

In [ ]:
img, _ = iter(loader).__next__()
img

In [ ]:
# detecting the faces again
face, prob = mtcnn0(img, return_prob= True)

print(face, prob)

In [ ]:
# the inception resnet required 4d tensors
try:
    resnet(face)
except ValueError as e:
    print(e)

In [ ]:
print(face.shape)

In [ ]:
# making it 4D
face_4D = face.unsqueeze(dim=0)

print('new shape', face_4D.shape)

In [ ]:
# now we can use inception resnet to get embeddings
embedding = resnet(face_4D)

print(f"Shape of face embedding: {embedding.shape}")

In [ ]:
idx_to_class.values()

In [ ]:
# run the resnet for all our images
name_to_embeddings = {name: [] for name in idx_to_class.values()}

for img, idx in loader:
    face, prob = mtcnn0(img, return_prob = True)
    if face != None and prob > 0.9:
        emb = resnet(face.unsqueeze(0))
        name_to_embeddings[idx_to_class[idx]].append(emb)

In [ ]:
# create an average embedding over the given images to represent the faceprint

# convert each list of embeddings to a 2D PyTorch
embeddings_stan = torch.stack(name_to_embeddings['stanley'])
embeddings_timz = torch.stack(name_to_embeddings['timmie'])

print(embeddings_stan.shape, embeddings_timz.shape)

In [ ]:
# taking the average value
average_embeddings_stan = torch.mean(embeddings_stan, dim=0)
average_embeddings_timz = torch.mean(embeddings_timz, dim=0)

print(average_embeddings_stan.shape, average_embeddings_timz.shape)

In [ ]:
average_embeddings_stan

In [ ]:
# saving the embeddings
embeddings_to_save = [(average_embeddings_stan, 'stan'), (average_embeddings_timz, 'timmie')]

torch.save(embeddings_to_save, 'embeddings.pt')

In [ ]:
# incase the embeddings are needed later
embedding_data = torch.load('embeddings.pt')

names = [name for _, name in embedding_data]
print(names)

## Testing the Model

In [ ]:
# Testing the model on new images

test_stan_path = r'c:\Users\HP\Documents\stant2.jpg'
test_timz_path = r'c:\Users\HP\Documents\timz2.jpg'

In [ ]:
test_stan = PIL.Image.open(test_stan_path)
test_stan

In [ ]:
mtcnn1 = MTCNN(image_size=240, keep_all=True, min_face_size=40)

In [ ]:
img_cropped_list, prob_list = mtcnn1(test_stan, return_prob = True)

print(img_cropped_list, prob_list)

In [ ]:
# incase of more than one face
for i, prob in enumerate(prob_list):
    if prob > 0.9:
        emb = resnet(img_cropped_list[i].unsqueeze(0))

In [ ]:
emb.shape

In [ ]:
# calculate the distance from the average embedding
distances = {}
for known_emb, name in embedding_data:
    dist = torch.dist(emb, known_emb).item()
    distances[name] = dist

closest, min_dist = min(distances.items(), key=lambda x: x[1])

print(f"Closest match: {closest}")
print(f"Calculated distance: {min_dist :.2f}")

In [ ]:
# to get bounding boxes
boxes, _ = mtcnn1.detect(test_stan)

In [ ]:
boxes.shape

In [ ]:
threshold = 0.8    # for accuracy of detecting faces

# This sets the image size and draws the original image
width, height = test_stan.size
dpi = 96
fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)
axis = fig.subplots()
axis.imshow(test_stan)
plt.axis("off")

for box in boxes:
    rect = plt.Rectangle(
        (box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, color="blue"
    )
    axis.add_patch(rect)

    closest, min_dist = min(distances.items(), key=lambda x: x[1])

    # Drawing the box with recognition results

    if min_dist < threshold:
        name = closest
        color = "blue"
    else:
        name = "Unrecognized"
        color = "red"

    plt.text(
        box[0],
        box[1],
        f"{name} {min_dist:.2f}",
        fontsize=20,
        color=color,
        ha="left",
        va="bottom",
    )

plt.axis("off")
plt.show()

In [ ]:
# images with multiple persons
multiple_path = r'c:\Users\HP\Documents\multiple.jpg'
img_multiple = PIL.Image.open(multiple_path)

img_multiple

In [ ]:
# def a fn to recognize faces
def recognize_faces(img_path, embedding_data, mtcnn, resnet, threshold=0.7):
    # Generating the bounding boxes, faces tensors, and probabilities
    image = PIL.Image.open(img_path)
    boxes, probs = mtcnn.detect(image)
    cropped_images = mtcnn(image)

    if boxes is None:
        return

    # This sets the image size and draws the original image
    width, height = image.size
    dpi = 96
    fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)
    axis = fig.subplots()
    axis.imshow(image)
    plt.axis("off")

    # Iterating over each face and comparing it against the pre-calculated embeddings
    # from our "database"
    for box, prob, face in zip(boxes, probs, cropped_images):
        if prob < 0.90:
            continue

        # Draw bounding boxes for all detected faces
        rect = plt.Rectangle(
            (box[0], box[1]),
            box[2] - box[0],
            box[3] - box[1],
            fill=False,
            color="blue",
        )
        axis.add_patch(rect)

        # Find the closest face from our database of faces
        emb = resnet(face.unsqueeze(0))
        distances = {}
        for known_emb, name in embedding_data:
            dist = torch.dist(emb, known_emb).item()
            distances[name] = dist

        closest, min_dist = min(distances.items(), key=lambda x: x[1])

        # Drawing the box with recognition results
        name = closest if min_dist < threshold else "Unrecognized"
        color = "red" if name == "Unrecognized" else "blue"
        label = f"{name} {min_dist:.2f}"

        axis.text(box[0], box[1], label, fontsize=20, color=color)

    plt.axis("off")
    plt.show()

In [ ]:
recognize_faces(img_path=multiple_path, embedding_data=embedding_data,mtcnn=mtcnn1, resnet=resnet, threshold=0.75)